# Plant Disease Classification Using Deep Learning and Transfer Learning

**CENG 476 – Introduction to Deep Learning**  
**Student Name:** Emir EVREN  
**Student ID:** 210444038

This notebook presents the main experimental workflow and the final evidence used in the project. It covers the PlantVillage data split, the custom CNN baseline, transfer learning with ResNet18 and EfficientNet-B0, learning-rate selection, dropout ablation, regularization, final test metrics, ROC-AUC, ensemble evaluation, and Grad-CAM analysis.

## 1. Project Setup

The notebook is designed to work when opened either from the repository root or directly from the `notebooks/` directory. The heavy training runs are not repeated automatically; saved histories, evaluation files, and figures in `outputs/` are used to reproduce the analysis.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Outputs found:", OUTPUT_DIR.exists())

## 2. Dataset and Split

The project uses the **PlantVillage** leaf-image dataset with **54,305 RGB images across 38 classes**. Images are processed to `3 × 224 × 224`.

| Split | Images | Purpose |
|---|---:|---|
| Train | 43,444 | Optimization and training augmentation |
| Validation | 5,430 | Scheduler, checkpoint selection, and hyperparameter decisions |
| Locked test | 5,431 | Final evaluation only |

The original validation directory is divided 50/50 with a **stratified split using seed 42**. Validation and test indices are checked to be disjoint. Because the class distribution is imbalanced, **Macro-F1** is the main validation metric for model selection.

Training augmentation:
- Random resized crop to 224, scale 0.80–1.00
- Horizontal flip with probability 0.50
- Rotation within ±15°
- Brightness/contrast/saturation jitter of 0.20
- ImageNet normalization

Validation and test preprocessing is deterministic: resize to 256, center crop to 224, tensor conversion, and ImageNet normalization.

In [ ]:
class_distribution_path = OUTPUT_DIR / "class_distribution.csv"
if class_distribution_path.exists():
    class_distribution = pd.read_csv(class_distribution_path)
    display(class_distribution.head())
else:
    print("Class distribution CSV not found.")

## 3. Custom Baseline CNN

The baseline is trained from scratch and contains four convolutional feature-extraction blocks:

`Conv 3×3 → BatchNorm → ReLU → MaxPool`

Channel progression: `32 → 64 → 128 → 256`.

After the convolutional blocks, adaptive average pooling reduces the feature map to `1 × 1`, followed by dropout and a `Linear(256, 38)` classifier. The network has **399,142 trainable parameters**.

The model returns **raw logits**. `CrossEntropyLoss` expects logits and applies the required log-softmax operation internally, so an explicit Softmax layer is not placed before the training loss.

In [ ]:
architecture_figure = OUTPUT_DIR / "figures" / "baseline_cnn_architecture.png"
if architecture_figure.exists():
    display(Image(filename=str(architecture_figure), width=1100))
else:
    print("Architecture figure not found:", architecture_figure)

## 4. Transfer-Learning Models

Two ImageNet-pretrained architectures are fine-tuned end-to-end:

- **ResNet18:** classifier replaced by `Dropout(0.30) + Linear(512, 38)`.
- **EfficientNet-B0:** classifier replaced by `Dropout(0.30) + Linear(1280, 38)`.

ResNet18 uses ReLU activations, while EfficientNet-B0 uses its native SiLU activations. Differential learning rates are used during fine-tuning: the pretrained backbone is updated conservatively while the new classifier learns faster.

## 5. Optimization and Regularization

All models use **AdamW** with `betas=(0.9, 0.999)` and unweighted `CrossEntropyLoss`.

| Setting | Baseline CNN | ResNet18 | EfficientNet-B0 |
|---|---:|---:|---:|
| Batch size | 64 | 32 | 32 |
| Maximum epochs | 15 | 12 | 12 |
| Backbone learning rate | 5e-4 | 1e-4 | 1e-4 |
| Classifier learning rate | 5e-4 | 5e-4 | 5e-4 |
| Final dropout | 0.40 | 0.30 | 0.30 |
| Weight decay | 1e-4 | 1e-4 | 1e-4 |
| Early-stopping patience | 6 | 5 | 5 |

Regularization includes:
- Batch Normalization
- Dropout
- AdamW weight decay (`1e-4`)
- Data augmentation
- `ReduceLROnPlateau`
- Early-stopping logic
- Best-checkpoint retention
- A locked test protocol

`ReduceLROnPlateau` monitors validation loss with `factor=0.5`, `patience=2`, and `min_lr=1e-6`.

Early stopping monitors validation Macro-F1. In the final reported runs, the configured maximum epoch limit was reached before the patience condition terminated training. Best-checkpoint selection still preserved the strongest validation checkpoint independently.

## 6. Learning-Rate Pilot Experiment

Three learning rates were tested for the baseline CNN in three-epoch pilot runs. The selected full-run learning rate was `5e-4`, which provided a useful balance between convergence speed and validation stability.

In [ ]:
lr_histories = {
    "1e-3": OUTPUT_DIR / "histories" / "baseline_b64_lr1e3_pilot_history.csv",
    "5e-4": OUTPUT_DIR / "histories" / "baseline_b64_lr5e4_pilot_history.csv",
    "3e-4": OUTPUT_DIR / "histories" / "baseline_b64_lr3e4_pilot_history.csv",
}

lr_rows = []
for label, path in lr_histories.items():
    if path.exists():
        history = pd.read_csv(path)
        best_idx = history["validation_macro_f1"].idxmax()
        best = history.loc[best_idx]
        lr_rows.append({
            "learning_rate": label,
            "best_epoch": int(best["epoch"]),
            "best_validation_macro_f1": float(best["validation_macro_f1"]),
            "best_validation_loss": float(best["validation_loss"]),
        })

lr_comparison = pd.DataFrame(lr_rows)
display(lr_comparison)

## 7. Dropout Ablation

A controlled dropout ablation was added for the custom CNN. Learning rate, batch size, weight decay, data split, random seed, and epoch count were held fixed; only the dropout probability changed.

| Dropout | Best validation Macro-F1 | Best validation loss |
|---:|---:|---:|
| **0.20** | **0.4990** | **1.4561** |
| 0.40 | 0.4953 | 1.4768 |
| 0.60 | 0.4311 | 1.9552 |

`0.20` and `0.40` were very close in the short three-epoch pilot, while `0.60` learned more slowly and produced the weakest validation result. This suggests that a high dropout value can suppress learning too strongly in this setting. The test set was not used to select the dropout value.

The final baseline run remains the previously selected `Dropout(0.40)` configuration; the ablation is interpreted as a controlled pilot rather than a retroactive test-driven hyperparameter change.

In [ ]:
dropout_csv = OUTPUT_DIR / "comparison" / "dropout_ablation.csv"
if dropout_csv.exists():
    dropout_results = pd.read_csv(dropout_csv)
    display(dropout_results)
else:
    print("Dropout comparison CSV not found.")

In [ ]:
dropout_figure = OUTPUT_DIR / "figures" / "dropout_ablation_validation_macro_f1.png"
if dropout_figure.exists():
    display(Image(filename=str(dropout_figure), width=800))
else:
    print("Dropout ablation figure not found:", dropout_figure)

## 8. Training Curves and Generalization

Training and validation curves were recorded for every full run. A balanced clean-training subset containing **20 images per class (760 total)** was also evaluated without augmentation after each epoch. This provides a clearer train-versus-validation comparison than the augmented training batches alone.

The baseline shows a larger generalization gap and more validation fluctuation. Transfer learning converges much more strongly and produces substantially better class-balanced validation performance.

In [ ]:
training_curve_files = {
    "Baseline CNN": OUTPUT_DIR / "figures" / "baseline_b64_lr5e4_full_training_curves.png",
    "ResNet18": OUTPUT_DIR / "figures" / "resnet18_b32_blr1e4_hlr5e4_full_training_curves.png",
    "EfficientNet-B0": OUTPUT_DIR / "figures" / "efficientnet_b0_b32_blr1e4_hlr5e4_full_training_curves.png",
}

for name, path in training_curve_files.items():
    print(name)
    if path.exists():
        display(Image(filename=str(path), width=900))
    else:
        print("Figure not found:", path)

## 9. Final Individual-Model Results

The locked test set is used only after model and checkpoint selection.

| Model | Parameters | Val. Macro-F1 | Test Accuracy | Test Macro-F1 | Weighted-F1 | Errors |
|---|---:|---:|---:|---:|---:|---:|
| Baseline CNN | 399,142 | 0.8121 | 87.42% | 0.8201 | 0.8711 | 683 |
| ResNet18 | 11,196,006 | 0.9883 | 99.26% | 0.9869 | 0.9927 | 40 |
| EfficientNet-B0 | 4,056,226 | **0.9924** | **99.52%** | **0.9923** | **0.9952** | **26** |

EfficientNet-B0 is the strongest individual model and achieves better accuracy with substantially fewer parameters than ResNet18.

In [ ]:
model_comparison_path = OUTPUT_DIR / "comparison" / "model_comparison.csv"
if model_comparison_path.exists():
    model_comparison = pd.read_csv(model_comparison_path)
    columns = [
        "model", "parameters", "best_epoch", "validation_macro_f1",
        "test_accuracy", "test_macro_precision", "test_macro_recall",
        "test_macro_f1", "test_weighted_f1", "test_errors"
    ]
    display(model_comparison[columns])
else:
    print("Model comparison CSV not found.")

In [ ]:
comparison_figure = OUTPUT_DIR / "figures" / "final_model_comparison.png"
if comparison_figure.exists():
    display(Image(filename=str(comparison_figure), width=950))
else:
    print("Model comparison figure not found:", comparison_figure)

## 10. ROC-AUC Analysis

A macro/micro one-vs-rest ROC figure was generated for the strongest individual model, EfficientNet-B0.

- **Macro-average ROC-AUC:** 0.999986
- **Micro-average ROC-AUC:** 0.999990

The macro average gives equal importance to every class, while the micro average aggregates all one-vs-rest decisions. The 38 per-class curves are not drawn on the same figure because that would make the visualization difficult to read.

In [ ]:
roc_figure = (
    OUTPUT_DIR / "figures" /
    "efficientnet_b0_b32_blr1e4_hlr5e4_full_test_roc_curve.png"
)
if roc_figure.exists():
    display(Image(filename=str(roc_figure), width=850))
else:
    print("ROC figure not found:", roc_figure)

## 11. Soft-Voting Ensemble

ResNet18 and EfficientNet-B0 were combined with weighted soft voting. Candidate weights were selected **only using validation Macro-F1**. The best eligible combination was a 50:50 mixture.

| Model | Test Accuracy | Test Macro-F1 | Weighted-F1 | Errors |
|---|---:|---:|---:|---:|
| EfficientNet-B0 | 99.52% | 0.9923 | 0.9952 | 26 |
| ResNet18 + EfficientNet-B0 | **99.76%** | **0.9950** | **0.9976** | **13** |

The ensemble reduces the EfficientNet-B0 error count from 26 to 13.

In [ ]:
ensemble_cm = (
    OUTPUT_DIR / "figures" /
    "resnet18_efficientnet_soft_voting_test_confusion_matrix.png"
)
if ensemble_cm.exists():
    display(Image(filename=str(ensemble_cm), width=1000))
else:
    print("Ensemble confusion matrix not found:", ensemble_cm)

## 12. Grad-CAM Explainability

Grad-CAM was applied to EfficientNet-B0 to inspect where the model concentrates its activation. Correct predictions generally focus on lesions, discoloration, venation, and leaf texture. Error examples often involve visually similar disease patterns.

Grad-CAM is used as a qualitative interpretation tool rather than evidence of causal reasoning.

In [ ]:
gradcam_figures = [
    OUTPUT_DIR / "figures" / "efficientnet_b0_gradcam_correct_examples.png",
    OUTPUT_DIR / "figures" / "efficientnet_b0_gradcam_error_examples.png",
]

for path in gradcam_figures:
    if path.exists():
        display(Image(filename=str(path), width=1000))
    else:
        print("Grad-CAM figure not found:", path)

## 13. Main Conclusions

1. The scratch CNN provides a useful baseline but is clearly weaker than the pretrained models.
2. Transfer learning produces the largest improvement in accuracy and Macro-F1.
3. EfficientNet-B0 is the best individual model and is more parameter-efficient than ResNet18.
4. The dropout pilot shows that very strong dropout (`0.60`) slows learning; `0.20` and `0.40` are close over three epochs.
5. Macro/micro ROC-AUC values are both approximately 1.0 for EfficientNet-B0 on the locked test split.
6. A 50:50 ResNet18/EfficientNet-B0 soft-voting ensemble further reduces the remaining errors.
7. Grad-CAM provides qualitative evidence that predictions usually rely on relevant leaf regions.
8. PlantVillage is largely a controlled-background dataset, so these results should not be interpreted as guaranteed field performance.

## 14. Reproduction Commands

The commands below reproduce the main training and final analysis stages from the repository root.

```powershell
.\.venv\Scripts\python.exe src\download_data.py
.\.venv\Scripts\python.exe src\data_setup.py
.\.venv\Scripts\python.exe src\sanity_check.py

.\.venv\Scripts\python.exe src\train_baseline.py --epochs 15 --learning-rate 0.0005 --batch-size 64 --num-workers 2 --weight-decay 0.0001 --dropout 0.4 --run-name baseline_b64_lr5e4_full

.\.venv\Scripts\python.exe src\train_resnet18.py --epochs 12 --batch-size 32 --num-workers 2 --backbone-learning-rate 0.0001 --classifier-learning-rate 0.0005 --weight-decay 0.0001 --dropout 0.3 --run-name resnet18_b32_blr1e4_hlr5e4_full

.\.venv\Scripts\python.exe src\train_efficientnet.py --epochs 12 --batch-size 32 --num-workers 2 --backbone-learning-rate 0.0001 --classifier-learning-rate 0.0005 --weight-decay 0.0001 --dropout 0.3 --run-name efficientnet_b0_b32_blr1e4_hlr5e4_full

.\.venv\Scripts\python.exe src\analyze_dropout_experiments.py
.\.venv\Scripts\python.exe src\generate_architecture_diagram.py
.\.venv\Scripts\python.exe src\generate_roc_curves.py --checkpoint outputs\checkpoints\efficientnet_b0_b32_blr1e4_hlr5e4_full_best.pt --batch-size 32
.\.venv\Scripts\python.exe src\compare_models.py
.\.venv\Scripts\python.exe src\evaluate_ensemble.py --batch-size 32
.\.venv\Scripts\python.exe src\generate_gradcam.py --batch-size 32 --num-correct 6 --num-errors 6
```